# Experiment B - MPJPE vs flow_period (Flow+KF only)

Loads all generated CSVs across all sequences and computes:
- per-sequence MPJPE
- dataset-level MPJPE (weighted by number of frames)

Fixed condition:
- `network_period = ?? s` (?? Hz)

Outputs:
- `results/summary/mpjpe_per_sequence.csv`
- `results/summary/mpjpe_table.csv`
- `plots/mpjpe_vs_flow_period.png`

Paper sentence:

> “We fix the detection period to 50 ms (20 Hz) in the flow-period study because it yields 10 output steps between detections at 200 Hz, creating a non-trivial interpolation interval while avoiding the long-horizon drift regime.”

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from datasets.utils import constants as ds_constants, parsing as ds_parsing


In [2]:
EXP_DIR = Path('/home/moveEnetFlow/experiments/expB_accuracy_vs_flow_period')
RAW_DIR = EXP_DIR / 'results' / 'raw'
SUMMARY_DIR = EXP_DIR / 'results' / 'summary'
PLOTS_DIR = EXP_DIR / 'plots'

# Must match the --data_root used in run_experiment_b.sh
DATA_ROOT = Path('/data/moveEnet_test/raw')

SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Flow periods to test (must match FLOW_PERIODS in run_experiment_b.sh)
period_tokens = ['0.001', '0.002', '0.005', '0.01', '0.02', '0.05', '0.1']
period_values = [float(x) for x in period_tokens]

print('DATA_ROOT:', DATA_ROOT)
print('RAW_DIR:', RAW_DIR)
print('Flow periods:', period_values)
print('Fixed network_period (s):', 0.1)

PermissionError: [Errno 13] Permission denied: '/home/moveEnetFlow'

In [ ]:
joint_names = list(ds_constants.HPECoreSkeleton.KEYPOINTS_MAP.keys())
assert len(joint_names) == 13, f'Expected 13 joints, got {len(joint_names)}'

def period_to_tag(period_token: str) -> str:
    """Convert period string to filename-safe tag (e.g., '0.001' -> '0p001')."""
    return period_token.replace('.', 'p')

def tag_to_period(tag: str) -> float:
    """Convert filename tag back to period value (e.g., '0p001' -> 0.001)."""
    return float(tag.replace('p', '.'))

def load_pred_eval_csv(csv_path: Path):
    """Load prediction CSV and extract timestamps + 13x2 joint coordinates."""
    arr = np.loadtxt(csv_path, delimiter=',', skiprows=1)
    if arr.ndim == 1:
        arr = arr[None, :]
    if arr.shape[1] < 28:
        raise ValueError(f'Unexpected CSV format for {csv_path}, columns={arr.shape[1]}')
    ts = arr[:, 0]
    joints = arr[:, 2:28].reshape(-1, 13, 2)
    return ts, joints

def interp_gt_to_pred_ts(gt_data, ts_pred: np.ndarray):
    """Interpolate GT joints to prediction timestamps."""
    ts_gt_raw = np.asarray(gt_data['ts'])
    ts_gt = np.concatenate(([0.0], ts_gt_raw, [ts_gt_raw[-1] + 1.0]))
    joints_gt = np.zeros((len(ts_pred), 13, 2), dtype=np.float64)
    for j, joint in enumerate(joint_names):
        x_raw = gt_data[joint][:, 0]
        y_raw = gt_data[joint][:, 1]
        x = np.concatenate(([x_raw[0]], x_raw, [x_raw[-1]]))
        y = np.concatenate(([y_raw[0]], y_raw, [y_raw[-1]]))
        joints_gt[:, j, 0] = np.interp(ts_pred, ts_gt, x)
        joints_gt[:, j, 1] = np.interp(ts_pred, ts_gt, y)
    return joints_gt

def extract_metadata_from_name(csv_path: Path):
    """Expected format: sequence__moveenet_ofk_fp_<tag>.csv"""
    name = csv_path.name
    m = re.match(r'(.+)__(moveenet_ofk)_fp_(\d+p\d+|\d+)\.csv$', name)
    if not m:
        raise ValueError(f'Unrecognized filename format: {name}')
    seq_key, _, fp_tag = m.groups()
    method = 'MoveEnet + OFK'
    flow_period_s = tag_to_period(fp_tag)
    return seq_key, method, flow_period_s

def gt_path_from_seq_key(seq_key: str):
    """Map encoded sequence key back to GT skeleton file path."""
    rel = seq_key.replace('__', '/') if '__' in seq_key else seq_key
    return DATA_ROOT / rel / 'ch0GT200Hzskeleton' / 'data.log'

In [ ]:
csv_files = sorted(RAW_DIR.glob('fp_*/*.csv'))
print(f'Found {len(csv_files)} prediction CSV files')
if len(csv_files) == 0:
    raise RuntimeError('No CSV files found. Run run_experiment_b.sh first.')

gt_cache = {}
rows = []

for csv_path in csv_files:
    seq_key, method, flow_period_s = extract_metadata_from_name(csv_path)

    gt_path = gt_path_from_seq_key(seq_key)
    if not gt_path.exists():
        print(f'Skipping (missing GT): {gt_path}')
        continue

    if gt_path not in gt_cache:
        gt_cache[gt_path] = ds_parsing.import_yarp_skeleton_data(gt_path, multi_channel=False)
    gt_data = gt_cache[gt_path]

    ts_pred, joints_pred = load_pred_eval_csv(csv_path)
    joints_gt = interp_gt_to_pred_ts(gt_data, ts_pred)

    err = np.linalg.norm(joints_pred - joints_gt, axis=2)  # [N,13]
    per_joint = err.mean(axis=0)
    mpjpe_px = float(per_joint.mean())
    n_samples = int(err.shape[0])

    rows.append({
        'sequence': seq_key,
        'method': method,
        'flow_period_s': flow_period_s,
        'flow_rate_hz': 1.0 / flow_period_s,
        'network_period_s': 0.05,
        'detection_rate_hz': 20.0,
        'n_samples': n_samples,
        'sum_err_px': float(err.sum()),
        'n_points': int(err.size),
        'mpjpe_px': mpjpe_px,
        **{f'mpjpe_{joint}': float(per_joint[i]) for i, joint in enumerate(joint_names)}
    })

df_seq = pd.DataFrame(rows).sort_values(['flow_period_s', 'sequence']).reset_index(drop=True)
print('Per-sequence rows:', len(df_seq))

hidden_cols = ['n_samples', 'sum_err_px', 'n_points']
visible_df = df_seq.drop(columns=hidden_cols, errors='ignore')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(visible_df)

In [ ]:
if len(df_seq) == 0:
    raise RuntimeError('No valid sequence rows to aggregate.')

agg = (
    df_seq.groupby(['flow_period_s', 'flow_rate_hz', 'method'], as_index=False)
    .agg(sum_err_px=('sum_err_px', 'sum'), n_points=('n_points', 'sum'), n_samples=('n_samples', 'sum'))
)
agg['mpjpe_px'] = agg['sum_err_px'] / agg['n_points']
df_table = agg.sort_values(['flow_period_s']).reset_index(drop=True)

per_seq_path = SUMMARY_DIR / 'mpjpe_per_sequence.csv'
table_path = SUMMARY_DIR / 'mpjpe_table.csv'
df_seq.to_csv(per_seq_path, index=False)
df_table.to_csv(table_path, index=False)

print('Saved:', per_seq_path)
print('Saved:', table_path)

display(df_table[['flow_period_s', 'flow_rate_hz', 'method', 'mpjpe_px']])

In [ ]:
plt.figure(figsize=(8, 5))
d = df_table[df_table['method'] == 'MoveEnet + OFK'].sort_values('flow_period_s')

plt.plot(d['flow_period_s'], d['mpjpe_px'], marker='o', label='MoveEnet + OFK')
plt.xscale('log')
plt.xlabel('flow_period (s)')
plt.ylabel('Dataset MPJPE (px)')
plt.title('Experiment B: Dataset MPJPE vs flow_period (network_period = 0.05 s)')
plt.grid(True, alpha=0.3)
plt.legend()
plot1 = PLOTS_DIR / 'mpjpe_vs_flow_period.png'
plt.tight_layout()
plt.savefig(plot1, dpi=180)
plt.show()
print('Saved:', plot1)

In [ ]:
# Optional: inspect where MPJPE starts increasing sharply ("knee").
d = df_table[df_table['method'] == 'MoveEnet + OFK'].sort_values('flow_period_s').copy()
d['delta_mpjpe_px'] = d['mpjpe_px'].diff()
display(d[['flow_period_s', 'flow_rate_hz', 'mpjpe_px', 'delta_mpjpe_px']])